In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. **Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`**
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * 02-05. Tools
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-06. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Streaming Text and Events
## Three Top-Level `StreamEvent.type` Values
* From `Runner.run_streamed(...).stream_events()`
* https://openai.github.io/openai-agents-python/ref/stream_events/ 

**`raw_response_event`**
* Low-level streaming events from Responses API
    * Many `event.data.type` values
* Provides token/text deltas, such as `ResponseTextDeltaEvent` / `response.output_text.delta`
    * Used for the "someone is typing a response" effect

**`run_item_stream_event`**
* Higher-level Agents SDK events
* Occur when agent completes a run item
    * E.g., message, tool call, tool output, handoff, reasoning, MCP event

**`agent_updated_stream_event`**
* Occurs when the current agent changes, typically for a handoff
---

### Agents SDK `run_item_stream_event` Event Names

```text
message_output_created
handoff_requested
handoff_occured
tool_called
tool_search_called
tool_search_output_created
tool_output
reasoning_item_created
mcp_approval_requested
mcp_approval_response
mcp_list_tools
```
* **`handoff_occured` is intentionally misspelled** 
    * For backward compatibility
    * https://openai.github.io/openai-agents-python/streaming/ 

## Python Tutor with Streamed Rewsponses

In [ ]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent
from IPython.display import HTML, Markdown, display

# create an Agent
tutor = Agent(
    name='Python Tutor',
    model='gpt-5.5', # gpt-5.4-mini is the current default
    instructions="""You are a Python tutor for novice programmers. 
        Provide concise, simple answers to Python questions."""
)

In [ ]:
print('I am a Python tutor. How can I help you?') 
turn_count = 1 # turn counter for user-input prompts
previous_response_id = None # used to chain responses for context

# run until the user input is 'exit'
while ((prompt := input(f'\n Input [{turn_count}]: ')) != 'exit'):
    with trace(f'02-03-streaming-response-[{turn_count}]'):
        # launch agent asynchronously and stream its response
        result = Runner.run_streamed(tutor, prompt,
            # SDK should chain responses for multi-turn conversation state 
            previous_response_id=previous_response_id,
            auto_previous_response_id=True # support response chaining 
        )

        # blank line in conversation after user input
        display(HTML('<div style="height: 1em;"/>'))
        markdown_text = ''
        handle = display(Markdown(markdown_text), display_id=True)
        
        # print text deltas as they arrive, similar to ChatGPT streaming
        async for event in result.stream_events():
            # if the event is part of the text response, display it; 
            # provides effect of someone typing a response to you
            if (event.type == 'raw_response_event' and 
                isinstance(event.data, ResponseTextDeltaEvent)): 
                markdown_text += event.data.delta
                handle.update(Markdown(markdown_text))

        turn_count += 1 
        display(HTML('<hr>'))

        # save last response ID for context during the next turn
        previous_response_id = result.last_response_id

print('User terminated app.')

---

## Sample Inputs for This Demo
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
* Are there any other ways?

---

## Useful Inspection Pattern

```python
async for event in result.stream_events():
    print(event.type)

    if event.type == 'run_item_stream_event':
        print('  name:', event.name)
        print('  item type:', event.item.type)

    elif event.type == 'agent_updated_stream_event':
        print('  new agent:', event.new_agent.name)

    elif event.type == 'raw_response_event':
        print('  raw event:', event.data.type)
```


---
&copy; 1992-2026 by Deitel & Associates, Inc. and Pearson Education, Inc. All Rights Reserved. 